<a href="https://colab.research.google.com/github/PUEDAN1/PUEDAN_CSCC102-FILEHANDLING/blob/main/Lab9_File_Handling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSV FILE


In [1]:
!pip -q install reportlab

import json
import uuid
import html
import re
import io
import base64

from IPython.display import display, HTML

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak
)
from reportlab.lib.units import mm


# ============================================================
# SETTINGS
# ============================================================

# Your data has no "Units" column, so every subject uses this value.
# (If every subject has the same units, GWA = simple average of grades.)
DEFAULT_UNITS = 3

# Optional: override units for specific subject codes, e.g.
# UNITS_BY_CODE = {"CC103": 4, "FA101": 2}
UNITS_BY_CODE = {}


# ============================================================
# STUDENT DATA
# Each subject: (Code, Title, Grade, Percent, Teacher)
# ============================================================

STUDENTS_RAW = [

    {
        "name": "Mark Anthony Santos",
        "program": "BS Computer Science",
        "subjects": [
            ("CC103", "Data Structures & Algorithms", 1.25, 95, "Prof. Raymond Cruz"),
            ("MATH101", "Calculus 1", 1.75, 88, "Engr. Francis Del Rosario"),
            ("GE101", "Understanding the Self", 1.50, 92, "Dr. Maria Cristina Castro"),
            ("CS201", "Object-Oriented Programming", 1.25, 94, "Prof. Ernesto Javier"),
            ("GE104", "Readings in Philippine History", 1.50, 91, "Prof. Teresa Aquino"),
        ],
    },

    {
        "name": "Bea Bianca Reyes",
        "program": "BS Business Administration - Marketing",
        "subjects": [
            ("MKTG201", "Principles of Marketing", 1.50, 92, "Ms. Clarissa Fernandez"),
            ("ECON102", "Macroeconomics", 1.75, 89, "Prof. Manuel Castelo"),
            ("ACCT101", "Financial Accounting", 2.00, 86, "Mr. Ricardo Alonzo"),
            ("BUS105", "Business Communication", 1.25, 94, "Ms. Patricia Valdez"),
            ("GE102", "Contemporary World", 1.50, 90, "Dr. Evelyn Soriano"),
        ],
    },

    {
        "name": "Joshua Miguel Villanueva",
        "program": "BS Medical Technology / Medical Laboratory Science",
        "subjects": [
            ("CHEM101", "General Chemistry", 2.00, 86, "Dr. Aris Mendoza"),
            ("BIO105", "Human Anatomy & Physiology", 1.75, 89, "Dr. Evelyn Soriano"),
            ("MT102", "Principles of Medical Laboratory Science 1", 1.50, 92, "Dr. Maria Cristina Castro"),
            ("CHEM102", "Organic Chemistry", 2.25, 83, "Dr. Aris Mendoza"),
            ("GE106", "Art Appreciation", 1.25, 95, "Ms. Patricia Valdez"),
        ],
    },

    {
        "name": "Andrea Nicole Dizon",
        "program": "BA Communication",
        "subjects": [
            ("LIT102", "World Literature", 1.75, 89, "Prof. Teresa Aquino"),
            ("COMM101", "Introduction to Mass Communication", 1.25, 94, "Ms. Clarissa Fernandez"),
            ("GE105", "Purposive Communication", 1.25, 95, "Ms. Patricia Valdez"),
            ("FIL101", "Sining ng Pakikipagtalastasan", 1.50, 91, "Prof. Teresa Aquino"),
            ("GE102", "Contemporary World", 1.75, 88, "Prof. Manuel Castelo"),
        ],
    },

    {
        "name": "Christian Joseph Ramos",
        "program": "BS Civil Engineering",
        "subjects": [
            ("MATH112", "Integral Calculus", 2.25, 83, "Engr. Francis Del Rosario"),
            ("PHYS101", "Physics for Engineers", 2.00, 85, "Engr. Francis Del Rosario"),
            ("CE101", "Engineering Drawing & Plans", 1.75, 88, "Engr. Roberto Ocampo"),
            ("CHEM101", "General Chemistry", 2.50, 80, "Dr. Aris Mendoza"),
            ("GE101", "Understanding the Self", 1.50, 91, "Dr. Maria Cristina Castro"),
        ],
    },

    {
        "name": "Samantha Grace Navarro",
        "program": "BS Nursing",
        "subjects": [
            ("BIO105", "Human Anatomy & Physiology", 1.50, 91, "Dr. Evelyn Soriano"),
            ("NURS101", "Fundamentals of Nursing Practice", 1.75, 88, "Ms. Clarissa Fernandez"),
            ("CHEM103", "Biochemistry for Health Sciences", 2.00, 86, "Dr. Aris Mendoza"),
            ("PSY101", "General Psychology", 1.25, 94, "Dr. Maria Cristina Castro"),
            ("GE104", "Readings in Philippine History", 1.50, 92, "Prof. Teresa Aquino"),
        ],
    },

    {
        "name": "Daniel Paolo Garcia",
        "program": "BS Accountancy",
        "subjects": [
            ("ECON102", "Macroeconomics", 2.50, 80, "Prof. Manuel Castelo"),
            ("ACCT101", "Financial Accounting & Reporting", 2.00, 85, "Mr. Ricardo Alonzo"),
            ("TAX101", "Income Taxation", 2.25, 82, "Mr. Ricardo Alonzo"),
            ("MATH105", "Mathematics in the Modern World", 1.75, 89, "Engr. Francis Del Rosario"),
            ("GE106", "Art Appreciation", 1.50, 91, "Ms. Patricia Valdez"),
        ],
    },

    {
        "name": "Alyssa Marie Mercado",
        "program": "BS Fine Arts / Design",
        "subjects": [
            ("GE106", "Art Appreciation", 1.25, 94, "Ms. Patricia Valdez"),
            ("FA101", "Drawing & Visual Representation", 1.00, 98, "Ms. Patricia Valdez"),
            ("FA102", "History of Art & Design", 1.50, 92, "Prof. Teresa Aquino"),
            ("COMM101", "Digital Media Production", 1.25, 95, "Ms. Clarissa Fernandez"),
            ("GE101", "Understanding the Self", 1.75, 88, "Dr. Maria Cristina Castro"),
        ],
    },

    {
        "name": "Gabriel Ryan Torres",
        "program": "BS Computer Engineering",
        "subjects": [
            ("CPE202", "Digital Systems & Logic Design", 1.75, 88, "Prof. Ernesto Javier"),
            ("CC103", "Data Structures & Algorithms", 1.50, 91, "Prof. Raymond Cruz"),
            ("MATH112", "Differential Equations", 2.00, 85, "Engr. Francis Del Rosario"),
            ("PHYS102", "University Physics II", 1.75, 89, "Engr. Francis Del Rosario"),
            ("GE102", "Contemporary World", 1.50, 92, "Prof. Manuel Castelo"),
        ],
    },

    {
        "name": "Katrina Isabel Roxas",
        "program": "BS Psychology",
        "subjects": [
            ("PSY101", "General Psychology", 1.50, 93, "Dr. Maria Cristina Castro"),
            ("BIO105", "Human Anatomy & Physiology", 1.75, 89, "Dr. Evelyn Soriano"),
            ("PSY102", "Developmental Psychology", 1.25, 95, "Dr. Maria Cristina Castro"),
            ("STAT101", "Psychological Statistics", 1.75, 88, "Engr. Francis Del Rosario"),
            ("GE105", "Purposive Communication", 1.25, 96, "Ms. Patricia Valdez"),
        ],
    },

]


# ============================================================
# BUILD RECORDS + VALIDATE
# ============================================================

students = []

for raw in STUDENTS_RAW:

    records = []

    for code, title, grade, percent, teacher in raw["subjects"]:

        if not (1.0 <= float(grade) <= 5.0):
            raise ValueError(
                f"Invalid grade {grade} for {raw['name']} ({code})."
            )

        records.append({
            "Code": code,
            "Description": title,
            "Units": float(UNITS_BY_CODE.get(code, DEFAULT_UNITS)),
            "Teacher": teacher,
            "Percent": percent,
            "Grade": float(grade)
        })

    total_units = sum(r["Units"] for r in records)
    weighted_total = sum(r["Grade"] * r["Units"] for r in records)

    students.append({
        "name": raw["name"],
        "program": raw["program"],
        "records": records,
        "total_units": total_units,
        # Philippine grading: 1.00 = best, 5.00 = worst
        "highest_grade": min(r["Grade"] for r in records),
        "lowest_grade": max(r["Grade"] for r in records),
        "gwa": weighted_total / total_units
    })


# ============================================================
# LEADERBOARD RANKING
# Lower GWA = better. Students with the same GWA (to 2 decimals)
# share the same rank, e.g. 1, 2, 3, 3, 5, 5, 7 ...
# ============================================================

for student in students:
    student["gwa_rounded"] = round(student["gwa"], 2)

for student in students:
    student["rank"] = 1 + sum(
        1 for other in students
        if other["gwa_rounded"] < student["gwa_rounded"]
    )

leaderboard_order = sorted(
    students,
    key=lambda s: (s["rank"], s["name"])
)


# ============================================================
# PDF STYLES
# ============================================================

RED = colors.HexColor("#D90429")
BLACK = colors.HexColor("#222222")
DARK_GRAY = colors.HexColor("#555555")
GRAY = colors.HexColor("#777777")
LIGHT_GRAY = colors.HexColor("#F5F5F5")
BORDER = colors.HexColor("#DDDDDD")
WHITE = colors.white

GOLD_TINT = colors.HexColor("#FFF3C4")
SILVER_TINT = colors.HexColor("#ECECEC")
BRONZE_TINT = colors.HexColor("#F6E3D0")

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "TitleStyle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=22,
    leading=26,
    textColor=RED,
    alignment=TA_CENTER,
    spaceAfter=8
)

subtitle_style = ParagraphStyle(
    "SubtitleStyle",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=9,
    leading=12,
    textColor=GRAY,
    alignment=TA_CENTER,
    spaceAfter=15
)

normal_style = ParagraphStyle(
    "NormalStyle",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=9,
    textColor=BLACK,
    leading=12
)

small_style = ParagraphStyle(
    "SmallStyle",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=8,
    textColor=DARK_GRAY,
    leading=10
)

small_center_style = ParagraphStyle(
    "SmallCenterStyle",
    parent=small_style,
    alignment=TA_CENTER
)

header_style = ParagraphStyle(
    "HeaderStyle",
    parent=normal_style,
    fontName="Helvetica-Bold",
    textColor=WHITE
)

header_center_style = ParagraphStyle(
    "HeaderCenterStyle",
    parent=header_style,
    alignment=TA_CENTER
)

grade_style = ParagraphStyle(
    "GradeStyle",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=9,
    leading=12,
    textColor=RED,
    alignment=TA_CENTER
)

rank_style = ParagraphStyle(
    "RankStyle",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=11,
    leading=14,
    textColor=BLACK,
    alignment=TA_CENTER
)

name_bold_style = ParagraphStyle(
    "NameBoldStyle",
    parent=normal_style,
    fontName="Helvetica-Bold"
)

summary_black_style = ParagraphStyle(
    "SummaryBlack",
    parent=normal_style,
    fontName="Helvetica-Bold",
    fontSize=16,
    leading=20,
    textColor=BLACK,
    alignment=TA_CENTER
)

summary_red_style = ParagraphStyle(
    "SummaryRed",
    parent=summary_black_style,
    textColor=RED
)

gwa_label_style = ParagraphStyle(
    "GWALabel",
    parent=normal_style,
    fontName="Helvetica-Bold",
    fontSize=10,
    leading=14,
    textColor=DARK_GRAY,
    alignment=TA_CENTER
)

gwa_style = ParagraphStyle(
    "GWAStyle",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=28,
    leading=34,
    textColor=RED,
    alignment=TA_CENTER
)

note_style = ParagraphStyle(
    "NoteStyle",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=8,
    leading=11,
    textColor=GRAY,
    alignment=TA_CENTER
)


# ============================================================
# PDF BUILDERS
# ============================================================

FULL_WIDTH = 176 * mm


def build_leaderboard_report():
    """Return the flowables for the LEADERBOARD page."""

    story = []

    story.append(Paragraph("GWA LEADERBOARD", title_style))
    story.append(
        Paragraph(
            "Ranked by General Weighted Average "
            "(1.00 = best, 5.00 = worst)",
            subtitle_style
        )
    )

    table_data = [[
        Paragraph("Rank", header_center_style),
        Paragraph("Student", header_style),
        Paragraph("Program / Major", header_style),
        Paragraph("Units", header_center_style),
        Paragraph("GWA", header_center_style)
    ]]

    for student in leaderboard_order:
        table_data.append([
            Paragraph(str(student["rank"]), rank_style),
            Paragraph(html.escape(student["name"]), name_bold_style),
            Paragraph(html.escape(student["program"]), small_style),
            Paragraph(f"{student['total_units']:g}", small_center_style),
            Paragraph(f"{student['gwa']:.2f}", grade_style)
        ])

    leaderboard_table = Table(
        table_data,
        colWidths=[16 * mm, 52 * mm, 62 * mm, 18 * mm, 28 * mm],
        repeatRows=1
    )

    table_style = [
        ("BACKGROUND", (0, 0), (-1, 0), RED),
        ("BACKGROUND", (0, 1), (-1, -1), WHITE),
        ("BOX", (0, 0), (-1, -1), 0.8, BORDER),
        ("INNERGRID", (0, 0), (-1, -1), 0.5, BORDER),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 5),
        ("RIGHTPADDING", (0, 0), (-1, -1), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]

    # Gold / silver / bronze tint for the top 3 ranks
    tints = {1: GOLD_TINT, 2: SILVER_TINT, 3: BRONZE_TINT}

    for row_number, student in enumerate(leaderboard_order, start=1):
        if student["rank"] in tints:
            table_style.append((
                "BACKGROUND",
                (0, row_number),
                (-1, row_number),
                tints[student["rank"]]
            ))

    leaderboard_table.setStyle(TableStyle(table_style))

    story.append(leaderboard_table)
    story.append(Spacer(1, 10))
    story.append(
        Paragraph(
            "Students with the same GWA share the same rank. "
            "Gold, silver and bronze rows mark ranks 1, 2 and 3.",
            note_style
        )
    )

    return story


def build_report(student):
    """Return the list of flowables for ONE student's report page."""

    story = []

    story.append(Paragraph("STUDENT GWA", title_style))
    story.append(Paragraph("General Weighted Average Report", subtitle_style))

    # ---- Student information ----
    student_info = [
        [
            Paragraph("<b>Student Name</b>", small_style),
            Paragraph(html.escape(student["name"]), normal_style)
        ],
        [
            Paragraph("<b>Program / Major</b>", small_style),
            Paragraph(html.escape(student["program"]), normal_style)
        ],
        [
            Paragraph("<b>Leaderboard Rank</b>", small_style),
            Paragraph(
                f"#{student['rank']} of {len(students)}",
                normal_style
            )
        ]
    ]

    student_table = Table(
        student_info,
        colWidths=[40 * mm, 136 * mm]
    )

    student_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), LIGHT_GRAY),
        ("BOX", (0, 0), (-1, -1), 0.8, BORDER),
        ("INNERGRID", (0, 0), (-1, -1), 0.5, BORDER),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 7),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
    ]))

    story.append(student_table)
    story.append(Spacer(1, 10))

    # ---- Subject table ----
    table_data = [[
        Paragraph("Code", header_style),
        Paragraph("Description", header_style),
        Paragraph("Teacher", header_style),
        Paragraph("Units", header_center_style),
        Paragraph("Percent", header_center_style),
        Paragraph("Grade", header_center_style)
    ]]

    for r in student["records"]:
        table_data.append([
            Paragraph(html.escape(r["Code"]), small_style),
            Paragraph(html.escape(r["Description"]), small_style),
            Paragraph(html.escape(r["Teacher"]), small_style),
            Paragraph(f"{r['Units']:g}", small_center_style),
            Paragraph(f"{r['Percent']}%", small_center_style),
            Paragraph(f"{r['Grade']:.2f}", grade_style)
        ])

    subject_table = Table(
        table_data,
        colWidths=[22 * mm, 56 * mm, 45 * mm, 15 * mm, 20 * mm, 18 * mm],
        repeatRows=1
    )

    subject_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), RED),
        ("BACKGROUND", (0, 1), (-1, -1), WHITE),
        ("BOX", (0, 0), (-1, -1), 0.8, BORDER),
        ("INNERGRID", (0, 0), (-1, -1), 0.5, BORDER),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 5),
        ("RIGHTPADDING", (0, 0), (-1, -1), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))

    story.append(subject_table)
    story.append(Spacer(1, 12))

    # ---- Summary ----
    summary_data = [
        [
            Paragraph("<b>Total Units</b>", small_style),
            Paragraph("<b>Highest Grade</b>", small_style),
            Paragraph("<b>Lowest Grade</b>", small_style)
        ],
        [
            Paragraph(f"{student['total_units']:g}", summary_black_style),
            Paragraph(f"{student['highest_grade']:.2f}", summary_red_style),
            Paragraph(f"{student['lowest_grade']:.2f}", summary_red_style)
        ]
    ]

    summary_table = Table(
        summary_data,
        colWidths=[FULL_WIDTH / 3] * 3
    )

    summary_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), LIGHT_GRAY),
        ("BOX", (0, 0), (-1, -1), 0.8, BORDER),
        ("INNERGRID", (0, 0), (-1, -1), 0.5, BORDER),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]))

    story.append(summary_table)
    story.append(Spacer(1, 12))

    # ---- GWA card ----
    gwa_data = [
        [Paragraph("COMPUTED GWA", gwa_label_style)],
        [Paragraph(f"{student['gwa']:.2f}", gwa_style)]
    ]

    gwa_table = Table(gwa_data, colWidths=[FULL_WIDTH])

    gwa_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), WHITE),
        ("BOX", (0, 0), (-1, -1), 1.5, RED),
        ("LINEABOVE", (0, 0), (-1, 0), 5, RED),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, 0), 10),
        ("BOTTOMPADDING", (0, 0), (-1, 0), 2),
        ("TOPPADDING", (0, 1), (-1, 1), 2),
        ("BOTTOMPADDING", (0, 1), (-1, 1), 12),
    ]))

    story.append(gwa_table)

    return story


def make_pdf_data_url(story, title):
    """Build a PDF from a story and return it as a base64 data URL."""

    buffer = io.BytesIO()

    doc = SimpleDocTemplate(
        buffer,
        pagesize=A4,
        rightMargin=15 * mm,
        leftMargin=15 * mm,
        topMargin=15 * mm,
        bottomMargin=15 * mm,
        title=title
    )

    doc.build(story)

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return "data:application/pdf;base64," + encoded


# One PDF per student
for student in students:

    student["pdf_url"] = make_pdf_data_url(
        build_report(student),
        f"GWA - {student['name']}"
    )

    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", student["name"]).strip("_")
    student["pdf_filename"] = f"{safe_name}_GWA.pdf"


# Leaderboard PDF (1 page)
leaderboard_pdf_url = make_pdf_data_url(
    build_leaderboard_report(),
    "GWA Leaderboard"
)


# One combined PDF: leaderboard first, then one page per student
combined_story = build_leaderboard_report()

for student in students:
    combined_story.append(PageBreak())
    combined_story.extend(build_report(student))

all_pdf_url = make_pdf_data_url(combined_story, "GWA - All Students")


# ============================================================
# GUI
# ============================================================

gui_id = "gwa_" + uuid.uuid4().hex[:8]

students_json = json.dumps(students, ensure_ascii=False)

html_code = r"""
<style>

#gwa___GUI_ID__ {
    font-family: Arial, Helvetica, sans-serif;
    max-width: 1100px;
    margin: 25px auto;
    color: #222222;
}

#gwa___GUI_ID__ * {
    box-sizing: border-box;
}

.gwa-app {
    background: #ffffff;
    border: 1px solid #dddddd;
    border-radius: 14px;
    overflow: hidden;
    box-shadow: 0 5px 20px rgba(0,0,0,0.08);
}

.gwa-header {
    background: #ffffff;
    padding: 25px 30px;
    border-bottom: 5px solid #D90429;
}

.gwa-header h1 {
    margin: 0;
    color: #D90429;
    font-size: 30px;
    font-weight: 800;
}

.gwa-header p {
    margin: 6px 0 0;
    color: #777777;
    font-size: 14px;
}

.gwa-content {
    padding: 25px;
}

.selector-row {
    display: flex;
    align-items: center;
    gap: 12px;
    flex-wrap: wrap;
    margin-bottom: 18px;
}

.selector-row label {
    font-size: 13px;
    font-weight: bold;
    color: #555555;
}

.selector-row select {
    flex: 1;
    min-width: 220px;
    padding: 10px 12px;
    font-size: 14px;
    border: 1px solid #DDDDDD;
    border-radius: 8px;
    background: #ffffff;
    color: #222222;
}

.info-box {
    background: #F5F5F5;
    border: 1px solid #DDDDDD;
    border-radius: 10px;
    padding: 18px;
    margin-bottom: 22px;
}

.info-grid {
    display: grid;
    grid-template-columns: 1fr 1fr 1fr;
    gap: 15px;
}

.info-item span {
    display: block;
    font-size: 12px;
    color: #777777;
    margin-bottom: 4px;
}

.info-item strong {
    font-size: 15px;
    color: #222222;
}

.section-title {
    color: #D90429;
    font-size: 18px;
    font-weight: bold;
    margin: 25px 0 12px;
}

.table-wrapper {
    width: 100%;
    overflow-x: auto;
    border: 1px solid #DDDDDD;
    border-radius: 10px;
}

.gwa-table {
    width: 100%;
    border-collapse: collapse;
    background: #ffffff;
}

.gwa-table th {
    background: #D90429;
    color: #ffffff;
    padding: 12px 10px;
    text-align: left;
    font-size: 13px;
}

.gwa-table td {
    padding: 11px 10px;
    border-bottom: 1px solid #eeeeee;
    font-size: 13px;
    color: #222222;
}

.gwa-table tr:last-child td {
    border-bottom: none;
}

.gwa-table td.grade {
    color: #D90429;
    font-weight: bold;
    text-align: center;
}

.gwa-table td.center,
.gwa-table th.center {
    text-align: center;
}

.button-row {
    display: flex;
    gap: 10px;
    margin-top: 20px;
    flex-wrap: wrap;
}

.calc-btn,
.print-btn,
.download-btn,
.download-all-btn {
    border: none;
    border-radius: 8px;
    padding: 12px 18px;
    font-size: 13px;
    font-weight: bold;
    cursor: pointer;
    transition: 0.2s;
    text-align: center;
    text-decoration: none;
    display: inline-block;
}

.calc-btn {
    background: #D90429;
    color: white;
}

.calc-btn:hover {
    background: #A80321;
}

.print-btn {
    background: #555555;
    color: white;
}

.print-btn:hover {
    background: #333333;
}

.download-btn {
    background: #D90429;
    color: white;
}

.download-btn:hover {
    background: #A80321;
}

.download-all-btn {
    background: #ffffff;
    color: #D90429;
    border: 2px solid #D90429;
}

.download-all-btn:hover {
    background: #fff0f2;
}

.result-section {
    display: none;
    margin-top: 25px;
}

.result-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 15px;
    margin-bottom: 18px;
}

.result-card {
    background: #F5F5F5;
    border: 1px solid #DDDDDD;
    border-radius: 10px;
    padding: 18px;
    text-align: center;
}

.result-card .label {
    font-size: 12px;
    color: #777777;
    margin-bottom: 7px;
}

.result-card .value {
    font-size: 25px;
    line-height: 1.3;
    font-weight: 800;
    color: #D90429;
}

.gwa-card {
    background: #ffffff;
    border: 2px solid #D90429;
    border-top: 7px solid #D90429;
    border-radius: 10px;
    padding: 20px 20px 24px;
    text-align: center;
}

.gwa-card .label {
    color: #555555;
    font-size: 13px;
    font-weight: bold;
}

.gwa-card .gwa-value {
    color: #D90429;
    font-size: 42px;
    line-height: 1.3;
    font-weight: 900;
    margin-top: 5px;
}

.rank-note {
    margin-top: 12px;
    font-size: 13px;
    color: #555555;
    font-weight: bold;
}

.hint {
    font-size: 12px;
    color: #777777;
    margin: -6px 0 12px;
}


/* ============================================================
   LEADERBOARD
   ============================================================ */

.leaderboard-section {
    margin-top: 35px;
    padding-top: 10px;
    border-top: 2px dashed #DDDDDD;
}

.leaderboard-title {
    color: #D90429;
    font-size: 24px;
    font-weight: 800;
    margin: 20px 0 4px;
    text-align: center;
}

.leaderboard-sub {
    text-align: center;
    color: #777777;
    font-size: 13px;
    margin-bottom: 22px;
}

/* ---------- Top 3 podium cards ---------- */

.podium {
    display: flex;
    justify-content: center;
    align-items: flex-end;
    gap: 14px;
    flex-wrap: wrap;
    margin-bottom: 26px;
}

.podium-card {
    flex: 1;
    min-width: 190px;
    max-width: 300px;
    background: #ffffff;
    border: 2px solid #DDDDDD;
    border-radius: 14px;
    padding: 18px 14px 20px;
    text-align: center;
    cursor: pointer;
    transition: transform 0.15s, box-shadow 0.15s;
}

.podium-card:hover {
    transform: translateY(-3px);
    box-shadow: 0 8px 18px rgba(0,0,0,0.12);
}

.podium-card.rank-1 {
    border-color: #F2B705;
    background: #FFF8DC;
    padding-top: 28px;
    padding-bottom: 30px;
}

.podium-card.rank-2 {
    border-color: #B8B8B8;
    background: #F6F6F6;
}

.podium-card.rank-3 {
    border-color: #CD7F32;
    background: #FBEEE2;
}

.podium-medal {
    font-size: 42px;
    line-height: 1.2;
}

.podium-card.rank-1 .podium-medal {
    font-size: 54px;
}

.podium-place {
    font-size: 12px;
    font-weight: bold;
    letter-spacing: 1px;
    color: #777777;
    margin-top: 2px;
}

.podium-name {
    font-size: 16px;
    font-weight: 800;
    color: #222222;
    margin: 8px 0 3px;
}

.podium-program {
    font-size: 12px;
    color: #777777;
    min-height: 30px;
}

.podium-gwa {
    font-size: 34px;
    font-weight: 900;
    color: #D90429;
    line-height: 1.3;
    margin-top: 8px;
}

.podium-gwa-label {
    font-size: 11px;
    color: #777777;
    font-weight: bold;
}

/* ---------- Full ranked list ---------- */

.lb-list {
    border: 1px solid #DDDDDD;
    border-radius: 12px;
    overflow: hidden;
}

.lb-head,
.lb-row {
    display: grid;
    grid-template-columns: 60px 1.4fr 1.2fr 80px;
    align-items: center;
    gap: 12px;
    padding: 12px 16px;
}

.lb-head {
    background: #D90429;
    color: #ffffff;
    font-size: 12px;
    font-weight: bold;
}

.lb-head .center,
.lb-row .center {
    text-align: center;
}

.lb-row {
    border-bottom: 1px solid #eeeeee;
    cursor: pointer;
    background: #ffffff;
}

.lb-row:last-child {
    border-bottom: none;
}

.lb-row:hover {
    background: #fff0f2;
}

.lb-row.active {
    background: #ffe3e8;
}

.lb-rank {
    width: 36px;
    height: 36px;
    border-radius: 50%;
    background: #EEEEEE;
    color: #555555;
    font-weight: 800;
    font-size: 15px;
    display: flex;
    align-items: center;
    justify-content: center;
    margin: 0 auto;
}

.lb-rank.rank-1 {
    background: #F2B705;
    color: #ffffff;
}

.lb-rank.rank-2 {
    background: #B8B8B8;
    color: #ffffff;
}

.lb-rank.rank-3 {
    background: #CD7F32;
    color: #ffffff;
}

.lb-name {
    font-size: 14px;
    font-weight: bold;
    color: #222222;
}

.lb-program {
    font-size: 12px;
    color: #777777;
    margin-top: 2px;
}

.lb-bar {
    height: 12px;
    background: #F0F0F0;
    border-radius: 6px;
    overflow: hidden;
}

.lb-bar-fill {
    height: 100%;
    background: linear-gradient(90deg, #A80321, #D90429);
    border-radius: 6px;
}

.lb-row.rank-1 .lb-bar-fill {
    background: linear-gradient(90deg, #E0A800, #F2B705);
}

.lb-gwa {
    text-align: center;
    font-size: 18px;
    font-weight: 900;
    color: #D90429;
}

.lb-download {
    margin-top: 18px;
    text-align: center;
}

@media (max-width: 700px) {

    .info-grid,
    .result-grid {
        grid-template-columns: 1fr;
    }

    .gwa-header h1 {
        font-size: 24px;
    }

    .gwa-content {
        padding: 15px;
    }

    .lb-head,
    .lb-row {
        grid-template-columns: 48px 1fr 64px;
    }

    .lb-bar-col,
    .lb-bar {
        display: none;
    }
}


/* ============================================================
   PRINT (prints only the selected student's report)
   ============================================================ */

@media print {

    body {
        background: white !important;
    }

    #gwa___GUI_ID__ {
        max-width: none !important;
        margin: 0 !important;
    }

    .gwa-app {
        border: none !important;
        box-shadow: none !important;
    }

    .gwa-header {
        border-bottom: 5px solid #D90429 !important;
    }

    .button-row,
    .selector-row,
    .leaderboard-section {
        display: none !important;
    }

    .gwa-table th {
        background: #D90429 !important;
        color: white !important;
    }

    .gwa-card {
        border-color: #D90429 !important;
        border-top-color: #D90429 !important;
        break-inside: avoid;
        page-break-inside: avoid;
    }

    .result-grid,
    .result-card {
        break-inside: avoid;
        page-break-inside: avoid;
    }

    * {
        -webkit-print-color-adjust: exact !important;
        print-color-adjust: exact !important;
    }
}

</style>


<div id="gwa___GUI_ID__">

    <div class="gwa-app">

        <div class="gwa-header">
            <h1>STUDENT GWA</h1>
            <p>General Weighted Average Calculator</p>
        </div>

        <div class="gwa-content">

            <div class="selector-row">
                <label for="studentSelect___GUI_ID__">Select Student</label>
                <select
                    id="studentSelect___GUI_ID__"
                    onchange="selectStudent___GUI_ID__(this.value)"
                ></select>
            </div>

            <div class="info-box">
                <div class="info-grid">

                    <div class="info-item">
                        <span>Student Name</span>
                        <strong id="infoName___GUI_ID__"></strong>
                    </div>

                    <div class="info-item">
                        <span>Program / Major</span>
                        <strong id="infoProgram___GUI_ID__"></strong>
                    </div>

                    <div class="info-item">
                        <span>Leaderboard Rank</span>
                        <strong id="infoRank___GUI_ID__"></strong>
                    </div>

                </div>
            </div>


            <div class="section-title">
                Subject Records
            </div>

            <div class="table-wrapper">

                <table class="gwa-table">

                    <thead>
                        <tr>
                            <th>Code</th>
                            <th>Description</th>
                            <th>Teacher</th>
                            <th class="center">Units</th>
                            <th class="center">Percent</th>
                            <th class="center">Grade</th>
                        </tr>
                    </thead>

                    <tbody id="recordsBody___GUI_ID__">
                    </tbody>

                </table>

            </div>


            <div class="button-row">

                <button
                    class="calc-btn"
                    onclick="calculateGWA___GUI_ID__()"
                >
                    CALCULATE GWA
                </button>

                <button
                    class="print-btn"
                    id="printButton___GUI_ID__"
                    onclick="window.print()"
                    style="display:none;"
                >
                    🖨️ PRINT GWA
                </button>

                <a
                    class="download-btn"
                    id="downloadButton___GUI_ID__"
                    href="#"
                    download="GWA.pdf"
                    style="display:none;"
                >
                    📥 DOWNLOAD PDF
                </a>

                <a
                    class="download-all-btn"
                    href="__ALL_PDF_URL__"
                    download="All_Students_GWA.pdf"
                >
                    📚 DOWNLOAD ALL (1 PDF)
                </a>

            </div>


            <div
                class="result-section"
                id="result___GUI_ID__"
            >

                <div class="section-title">
                    GWA Results
                </div>

                <div class="result-grid">

                    <div class="result-card">
                        <div class="label">TOTAL UNITS</div>
                        <div class="value" id="totalUnits___GUI_ID__">0</div>
                    </div>

                    <div class="result-card">
                        <div class="label">HIGHEST GRADE</div>
                        <div class="value" id="highestGrade___GUI_ID__">0.00</div>
                    </div>

                    <div class="result-card">
                        <div class="label">LOWEST GRADE</div>
                        <div class="value" id="lowestGrade___GUI_ID__">0.00</div>
                    </div>

                </div>


                <div class="gwa-card">
                    <div class="label">COMPUTED GWA</div>
                    <div class="gwa-value" id="gwaValue___GUI_ID__">0.00</div>
                    <div class="rank-note" id="rankNote___GUI_ID__"></div>
                </div>

            </div>


            <!-- ================= LEADERBOARD ================= -->

            <div class="leaderboard-section">

                <div class="leaderboard-title">
                    🏆 GWA LEADERBOARD
                </div>

                <div class="leaderboard-sub">
                    Lower GWA = better (1.00 is the best). Same GWA = same rank.
                    Click a student to open their record.
                </div>

                <div class="podium" id="podium___GUI_ID__"></div>

                <div class="lb-list">

                    <div class="lb-head">
                        <div class="center">RANK</div>
                        <div>STUDENT</div>
                        <div class="lb-bar-col">STANDING (closer to 1.00 = fuller bar)</div>
                        <div class="center">GWA</div>
                    </div>

                    <div id="lbList___GUI_ID__"></div>

                </div>

                <div class="lb-download">
                    <a
                        class="download-btn"
                        href="__LEADERBOARD_PDF_URL__"
                        download="GWA_Leaderboard.pdf"
                    >
                        🏆 DOWNLOAD LEADERBOARD (PDF)
                    </a>
                </div>

            </div>

        </div>

    </div>

</div>


<script>

const students___GUI_ID__ = __STUDENTS_JSON__;

let current___GUI_ID__ = 0;


function escapeHTML___GUI_ID__(value) {

    return String(value)
        .replace(/&/g, "&amp;")
        .replace(/</g, "&lt;")
        .replace(/>/g, "&gt;")
        .replace(/"/g, "&quot;")
        .replace(/'/g, "&#039;");

}


function ordinal___GUI_ID__(n) {

    const s = ["th", "st", "nd", "rd"];
    const v = n % 100;

    return n + (s[(v - 20) % 10] || s[v] || s[0]);

}


function medal___GUI_ID__(rank) {

    if (rank === 1) { return "🥇"; }
    if (rank === 2) { return "🥈"; }
    if (rank === 3) { return "🥉"; }

    return "";

}


function computeStats___GUI_ID__(student) {

    let totalUnits = 0;
    let weightedTotal = 0;

    // Philippine grading: smaller number = better grade
    let highest = student.records[0].Grade;
    let lowest = student.records[0].Grade;

    student.records.forEach(function(record) {

        const units = Number(record.Units);
        const grade = Number(record.Grade);

        totalUnits += units;
        weightedTotal += grade * units;

        if (grade < highest) {
            highest = grade;
        }

        if (grade > lowest) {
            lowest = grade;
        }

    });

    return {
        totalUnits: totalUnits,
        highest: highest,
        lowest: lowest,
        gwa: weightedTotal / totalUnits
    };

}


function buildSelect___GUI_ID__() {

    const select =
        document.getElementById("studentSelect___GUI_ID__");

    select.innerHTML = "";

    students___GUI_ID__.forEach(function(student, index) {

        const option = document.createElement("option");

        option.value = index;
        option.textContent = (index + 1) + ". " + student.name;

        select.appendChild(option);

    });

}


function buildLeaderboard___GUI_ID__() {

    // Order by rank (ties: alphabetical)
    const ordered = students___GUI_ID__
        .map(function(student, index) {

            return {
                index: index,
                student: student,
                gwa: computeStats___GUI_ID__(student).gwa
            };

        })
        .sort(function(a, b) {

            if (a.student.rank !== b.student.rank) {
                return a.student.rank - b.student.rank;
            }

            return a.student.name.localeCompare(b.student.name);

        });


    // ---------- Podium: everyone ranked 1st, 2nd or 3rd ----------
    const podium =
        document.getElementById("podium___GUI_ID__");

    podium.innerHTML = "";

    ordered
        .filter(function(item) {
            return item.student.rank <= 3;
        })
        .forEach(function(item) {

            const rank = item.student.rank;

            const card = document.createElement("div");

            card.className = "podium-card rank-" + rank;

            card.innerHTML = `
                <div class="podium-medal">${medal___GUI_ID__(rank)}</div>
                <div class="podium-place">${ordinal___GUI_ID__(rank).toUpperCase()} PLACE</div>
                <div class="podium-name">${escapeHTML___GUI_ID__(item.student.name)}</div>
                <div class="podium-program">${escapeHTML___GUI_ID__(item.student.program)}</div>
                <div class="podium-gwa">${item.gwa.toFixed(2)}</div>
                <div class="podium-gwa-label">GWA</div>
            `;

            card.onclick = function() {
                selectStudent___GUI_ID__(item.index);
                window.scrollTo({ top: 0, behavior: "smooth" });
            };

            podium.appendChild(card);

        });


    // ---------- Full ranked list ----------
    const list =
        document.getElementById("lbList___GUI_ID__");

    list.innerHTML = "";

    ordered.forEach(function(item) {

        const rank = item.student.rank;

        // Bar: 1.00 = full bar, 5.00 = empty bar
        const percent = Math.max(
            0,
            Math.min(100, ((5 - item.gwa) / 4) * 100)
        );

        const row = document.createElement("div");

        row.className = "lb-row rank-" + (rank <= 3 ? rank : "other");
        row.setAttribute("data-index", item.index);

        row.innerHTML = `
            <div>
                <div class="lb-rank ${rank <= 3 ? "rank-" + rank : ""}">${rank}</div>
            </div>

            <div>
                <div class="lb-name">${escapeHTML___GUI_ID__(item.student.name)}</div>
                <div class="lb-program">${escapeHTML___GUI_ID__(item.student.program)}</div>
            </div>

            <div class="lb-bar">
                <div class="lb-bar-fill" style="width:${percent.toFixed(1)}%;"></div>
            </div>

            <div class="lb-gwa">${item.gwa.toFixed(2)}</div>
        `;

        row.onclick = function() {
            selectStudent___GUI_ID__(item.index);
            window.scrollTo({ top: 0, behavior: "smooth" });
        };

        list.appendChild(row);

    });

}


function highlightLeaderboard___GUI_ID__() {

    const rows = document.querySelectorAll(
        "#lbList___GUI_ID__ .lb-row"
    );

    rows.forEach(function(row) {

        if (Number(row.getAttribute("data-index")) === current___GUI_ID__) {
            row.classList.add("active");
        } else {
            row.classList.remove("active");
        }

    });

}


function selectStudent___GUI_ID__(index) {

    index = Number(index);

    current___GUI_ID__ = index;

    const student = students___GUI_ID__[index];

    document.getElementById(
        "studentSelect___GUI_ID__"
    ).value = index;

    document.getElementById(
        "infoName___GUI_ID__"
    ).textContent = student.name;

    document.getElementById(
        "infoProgram___GUI_ID__"
    ).textContent = student.program;

    document.getElementById(
        "infoRank___GUI_ID__"
    ).textContent =
        "#" + student.rank + " of " + students___GUI_ID__.length;

    const tbody =
        document.getElementById("recordsBody___GUI_ID__");

    tbody.innerHTML = "";

    student.records.forEach(function(record) {

        const row = document.createElement("tr");

        row.innerHTML = `
            <td>${escapeHTML___GUI_ID__(record.Code)}</td>
            <td>${escapeHTML___GUI_ID__(record.Description)}</td>
            <td>${escapeHTML___GUI_ID__(record.Teacher)}</td>
            <td class="center">${Number(record.Units).toString()}</td>
            <td class="center">${escapeHTML___GUI_ID__(record.Percent)}%</td>
            <td class="grade">${Number(record.Grade).toFixed(2)}</td>
        `;

        tbody.appendChild(row);

    });

    // New student selected: hide old results until Calculate is pressed
    document.getElementById(
        "result___GUI_ID__"
    ).style.display = "none";

    document.getElementById(
        "printButton___GUI_ID__"
    ).style.display = "none";

    const download =
        document.getElementById("downloadButton___GUI_ID__");

    download.style.display = "none";
    download.href = student.pdf_url;
    download.setAttribute("download", student.pdf_filename);

    highlightLeaderboard___GUI_ID__();

}


function calculateGWA___GUI_ID__() {

    const student = students___GUI_ID__[current___GUI_ID__];

    const stats = computeStats___GUI_ID__(student);

    document.getElementById(
        "totalUnits___GUI_ID__"
    ).textContent = Number(stats.totalUnits).toString();

    document.getElementById(
        "highestGrade___GUI_ID__"
    ).textContent = Number(stats.highest).toFixed(2);

    document.getElementById(
        "lowestGrade___GUI_ID__"
    ).textContent = Number(stats.lowest).toFixed(2);

    document.getElementById(
        "gwaValue___GUI_ID__"
    ).textContent = Number(stats.gwa).toFixed(2);

    document.getElementById(
        "rankNote___GUI_ID__"
    ).textContent =
        medal___GUI_ID__(student.rank) + " Leaderboard rank: " +
        ordinal___GUI_ID__(student.rank) + " of " +
        students___GUI_ID__.length;

    document.getElementById(
        "result___GUI_ID__"
    ).style.display = "block";

    document.getElementById(
        "printButton___GUI_ID__"
    ).style.display = "inline-block";

    document.getElementById(
        "downloadButton___GUI_ID__"
    ).style.display = "inline-block";

}


buildSelect___GUI_ID__();
buildLeaderboard___GUI_ID__();
selectStudent___GUI_ID__(0);

</script>
"""


# ============================================================
# REPLACE PLACEHOLDERS
# ============================================================

html_code = html_code.replace("__GUI_ID__", gui_id)
html_code = html_code.replace("__STUDENTS_JSON__", students_json)
html_code = html_code.replace("__ALL_PDF_URL__", all_pdf_url)
html_code = html_code.replace("__LEADERBOARD_PDF_URL__", leaderboard_pdf_url)


# ============================================================
# DISPLAY GUI
# ============================================================

display(HTML(html_code))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.2 MB/s eta 0:00:00


Code,Description,Teacher,Units,Percent,Grade
